In [ ]:
!pip install transformers torch pillow requests

In [ ]:
import torch
from transformers import SamModel, SamProcessor
from PIL import Image
import requests
import matplotlib.pyplot as plt
import numpy as np



Loading model...


In [11]:
print("Loading model...")
device = "cpu"
model = SamModel.from_pretrained("facebook/sam-vit-base").to(device)
processor = SamProcessor.from_pretrained("facebook/sam-vit-base")

Loading model...


In [ ]:

img_url = "../Dataset/dogs_vs_cats/test/cats/cat.10.jpg"
raw_image = Image.open(img_url).convert("RGB")


input_points = [[[400, 250]]]

print("Processing image and prompt...")

inputs = processor(raw_image, input_points=input_points, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)


print("Post-processing mask...")



Processing image and prompt...
Post-processing mask...
Post-processing mask...


In [ ]:
# Get the predicted masks and IoU scores
print("Output shapes:")
print(f"pred_masks shape: {outputs.pred_masks.shape}")
print(f"iou_scores shape: {outputs.iou_scores.shape}")


Output shapes:
pred_masks shape: torch.Size([1, 1, 3, 256, 256])
iou_scores shape: torch.Size([1, 1, 3])
Best mask index: 1


In [14]:
iou_scores = outputs.iou_scores.squeeze() 
best_mask_idx = torch.argmax(iou_scores)
print(f"Best mask index: {best_mask_idx}")

# Extract the predicted mask
predicted_mask = outputs.pred_masks.squeeze()[best_mask_idx].cpu().numpy()

Best mask index: 1


In [ ]:
def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6]) # Dodger blue
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

In [ ]:
fig, ax = plt.subplots()
ax.imshow(raw_image)
show_mask(predicted_mask, ax)
ax.plot(input_points[0][0][0], input_points[0][0][1], 'go') # 'go' = green dot
ax.axis('off')

output_filename = "sam_output.png"
plt.savefig(output_filename)
print(f"Done! Saved output to {output_filename}")
print(f"Test with a point at: {input_points[0][0]}")